### rag chain with faiss DB

In [3]:
import os
from dotenv import load_dotenv
import numpy as np
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import MessagesPlaceholder
#-------------------------------------------------------->
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_community.vectorstores import FAISS
#-------------------------------------------------------->
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models.base import init_chat_model
#-------------------------------------------------------->
from langchain_text_splitters import RecursiveCharacterTextSplitter
#-------------------------------------------------------->
from langchain_classic.chains import create_retrieval_chain
#-------------------------------------------------------->
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


load_dotenv()





True

#### Data ingestion and processing

In [6]:
data_loader=TextLoader("../data/python.txt", encoding="utf-8")
documents=data_loader.load()
print(documents)

[Document(metadata={'source': '../data/python.txt'}, page_content='\nPython Programming Introduction\n\nPython is a high-level programming language known for its simplicity and readability. \nIt was created by Guido van Rossum and released in 1991.\n\nKey Features:\n- Easy to learn and use\n- Large community support\n- Cross-platform compatibility\n\nPython is widely used in web development, data science, machine learning, and automation.\n\n\nMachine Learning Basics\n\nMachine learning is a branch of artificial intelligence that allows computers to learn \nfrom data without being explicitly programmed.\n\nTypes of Machine Learning:\n1. Supervised Learning\n2. Unsupervised Learning\n3. Reinforcement Learning\n\nApplications include image recognition, recommendation systems, and fraud detection.\n\n\nData Science Fundamentals\n\nData science combines statistics, programming, and domain knowledge to extract insights \nfrom data.\n\nKey Steps:\n- Data collection\n- Data cleaning\n- Data a

In [15]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks=text_splitter.split_documents(documents)
for doc in chunks:
    print(doc.page_content)
    print("\n---\n")


Python Programming Introduction

Python is a high-level programming language known for its simplicity and readability. 
It was created by Guido van Rossum and released in 1991.

---

Key Features:
- Easy to learn and use
- Large community support
- Cross-platform compatibility

Python is widely used in web development, data science, machine learning, and automation.

---

Machine Learning Basics

Machine learning is a branch of artificial intelligence that allows computers to learn 
from data without being explicitly programmed.

---

Types of Machine Learning:
1. Supervised Learning
2. Unsupervised Learning
3. Reinforcement Learning

Applications include image recognition, recommendation systems, and fraud detection.

---

Data Science Fundamentals

Data science combines statistics, programming, and domain knowledge to extract insights 
from data.

---

Key Steps:
- Data collection
- Data cleaning
- Data analysis
- Visualization

Tools used include Python, R, and SQL.

---



In [16]:
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1497.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### create faiss vectore store

In [18]:
vector_store=FAISS.from_documents(chunks, embedding_model)
print(f"Number of documents in vector store: {vector_store.index.ntotal}")

Number of documents in vector store: 6


In [19]:
vector_store.save_local("faiss_DB/faiss_index")

In [20]:
loaded_vector_store=FAISS.load_local("faiss_DB/faiss_index", embedding_model,allow_dangerous_deserialization=True)
print(f"Number of documents in loaded vector store: {loaded_vector_store.index.ntotal}")

Number of documents in loaded vector store: 6


In [21]:
query="What is Python?"
results=vector_store.similarity_search(query, k=3)
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result.page_content)
    print("\n---\n")


Result 1:
Python Programming Introduction

Python is a high-level programming language known for its simplicity and readability. 
It was created by Guido van Rossum and released in 1991.

---

Result 2:
Key Features:
- Easy to learn and use
- Large community support
- Cross-platform compatibility

Python is widely used in web development, data science, machine learning, and automation.

---

Result 3:
Key Steps:
- Data collection
- Data cleaning
- Data analysis
- Visualization

Tools used include Python, R, and SQL.

---



#### build rag chain with LCEL

In [22]:
load_dotenv()

True

In [23]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [24]:
llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0.7)

In [47]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that provides concise answers to questions based on the provided context.\nContext: {context}"),
    ("human", "{input}"),
])

In [48]:
retrieval=vector_store.as_retriever(search_kwargs={"k": 3},search_type="similarity")

In [49]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [54]:
rag_chain=(
    {
        "context":retrieval|format_docs,
        "input":RunnablePassthrough()
      

    }
    |prompt
    |llm
    |StrOutputParser()
)
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002785342EE90>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a helpful assistant that provides concise answers to questions based on the provided context.\nContext: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'vi

In [55]:
from langchain_classic.chains import create_history_aware_retriever
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that provides concise answers to questions based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),  # ✅ غيرنا من "question" إلى "input"
])
conversational_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retrieval,
    prompt=conversational_prompt  # ✅ دلوقتي prompt فيه "input" و "chat_history"
)


In [56]:
converstaional_rag_chain=(
    {
        "context":conversational_retriever|format_docs,
        "input":lambda x: x ["input"],
        "chat_history":lambda x: x ["chat_history"]
  }|
  prompt
  |llm
  |StrOutputParser()

)
chat_history=[]
    

In [57]:
response=rag_chain.invoke("what is python?")
print("Response:", response)

Response: Python is a high-level programming language known for its simplicity and readability. It was created by Guido van Rossum and released in 1991.


In [58]:
response1=converstaional_rag_chain.invoke(
    {
        "input":"what is python?",
        "chat_history":chat_history
    }
)

In [59]:
chat_history.append(HumanMessage(content="what is python?"))
chat_history.append(response1)

In [60]:
response2=converstaional_rag_chain.invoke(
    {
        "input":"what are its applications?",
        "chat_history":chat_history
    }
)
print("Response 1:", response1)
print("Response 2:", response2)

Response 1: Python is a high-level programming language known for its simplicity and readability. It was created by Guido van Rossum and released in 1991.
Response 2: Python's applications include:

1. Web development
2. Data science
3. Machine learning
4. Automation
